In [ ]:


EXPORT=False


YEAR_START = 2023
YEAR_END = 2023
SAMPLE_TYPE = 'TMAG'



import numpy as np
import pandas as pd
from pathlib import Path
pd.options.display.float_format = '{:.2f}'.format


PATH_SP = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Data' / 'Safe Equitable Resilient Infrastructure' / 'Travel Accessibility'
PATH_GIT = Path.home() / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
PATH_CODE    = PATH_GIT / 'Data' / 'Census'
PATH_CONFIG0 = PATH_GIT / 'config'
PATH_CONFIG  = PATH_CODE / 'config'

PATH_SERVER = Path(r'\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data')
PATH_ACCESS = Path(r'I:\Projects\Josh\Regional Monitoring\Accessibility')


import sys
sys.path.append(str(PATH_CONFIG0))
import functions as func



def process_access_1(df, geography, access, metric):

    df['Reference'] = '2024 overture, 2023 ACS'
    df['Year'] = 2023

    if geography == 'MPO':
        col = 'MPO_NAME'
    if geography == 'Community Type':
        col = 'Webmap_ComType'
    if geography == 'Jurisdiction':
        col = 'JURIS'
    if geography == 'County':
        col = 'name'
    df = df.rename(columns={col:geography})

    access_metrics = [f'transit_{access}', f'bike_{access}', f'walk_{access}']
    if access == 'emp':
        access_metrics .append(f'drive_{access}')
    df = df[['Reference', geography, 'Year'] + access_metrics]
    
    df = df.melt(id_vars=['Reference', geography, 'Year'], var_name='Mode', value_name=metric)

    if access == 'emp':
        conditions = [
            df['Mode'] == f'transit_{access}'
            , df['Mode'] == f'bike_{access}'
            , df['Mode'] == f'walk_{access}'
            , df['Mode'] == f'drive_{access}'
        ]
        choices = ['Public Transit', 'Bike', 'Walk', 'Drive']

    else:
        conditions = [
            df['Mode'] == f'transit_{access}'
            , df['Mode'] == f'bike_{access}'
            , df['Mode'] == f'walk_{access}'
        ]

        choices = ['Public Transit', 'Bike', 'Walk']

    df['Mode'] = np.select(conditions, choices, default='no')

    return df


def process_access_2(df, geography, access, metric):

    df['Reference'] = '2024 overture, 2023 ACS'
    df['Year'] = 2023

    if geography == 'MPO':
        col = 'MPO_NAME'
    if geography == 'Community Type':
        col = 'Webmap_ComType'
    if geography == 'Jurisdiction':
        col = 'JURIS'
    if geography == 'County':
        col = 'name'
    df = df.rename(columns={col:geography})

    access_metrics = [f'transit_{access}', f'bike_{access}', f'walk_{access}']
    if access == 'emp':
        access_metrics .append(f'drive_{access}')
    df = df[['Reference', geography, 'Race_Ethnicity', 'Year'] + access_metrics]

    df = df.melt(id_vars=['Reference', geography, 'Race_Ethnicity', 'Year'], var_name='Mode', value_name=metric)

    if access == 'emp':
        conditions = [
        df['Mode'] == f'transit_{access}'
        , df['Mode'] == f'bike_{access}'
        , df['Mode'] == f'walk_{access}'
        , df['Mode'] == f'drive_{access}'
        ]
        choices = ['Public Transit', 'Bike', 'Walk', 'Drive']

    else:
        conditions = [
            df['Mode'] == f'transit_{access}'
            , df['Mode'] == f'bike_{access}'
            , df['Mode'] == f'walk_{access}'
        ]
        choices = ['Public Transit', 'Bike', 'Walk']
    df['Mode'] = np.select(conditions, choices, default='no')

    return df


def export_transit(file_out, df, sample_type, indicator, year_start, year_end, geography):
    df_about = func.write_about(sample_type    = sample_type
                                , indicator    = indicator
                                , year_start   = year_start
                                , year_end     = year_end
                                , geography    = geography)
    if file_out.is_file():
        with pd.ExcelWriter(file_out, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
            df_about.to_excel(writer, sheet_name='About'  , index=False, header=False)
            df      .to_excel(writer, sheet_name=geography, index=False,             )
    else:
        with pd.ExcelWriter(file_out, engine='xlsxwriter') as writer:
            df_about.to_excel(writer, sheet_name='About'  , index=False, header=False)
            df      .to_excel(writer, sheet_name=geography, index=False,             )

        

TravelAccess_1

In [ ]:

# Indicator
indicator = 'TravelAccess_1'
folder = 'Mode'
geography = 'MPO'



## Employment ---
metric = 'Number of Jobs'
access = 'emp'
file_in = PATH_ACCESS / f"SACOG_MPO__access_pop_{access}.csv"
df_emp = pd.read_csv(file_in)
df_emp = process_access_1(df_emp, geography, access, metric)
display(df_emp.head())



## Education ---
metric = 'Number of Schools'
access = 'edu'
file_in = PATH_ACCESS / f"SACOG_MPO__access_pop_{access}.csv"
df_edu = pd.read_csv(file_in)
df_edu = process_access_1(df_edu, geography, access, metric)
display(df_edu.head())



## Neighborhood Services ---
metric = 'Number of Neighborhood Services'
access = 'nonwork'
file_in = PATH_ACCESS / f"SACOG_MPO__access_pop_{access}.csv"
df_nonwork = pd.read_csv(file_in)
df_nonwork = process_access_1(df_nonwork, geography, access, metric)
display(df_nonwork.head())



## Merge ---

df_mode = df_emp .merge(df_edu    , on=['Reference', geography, 'Year', 'Mode'])
df_mode = df_mode.merge(df_nonwork, on=['Reference', geography, 'Year', 'Mode'])


if EXPORT:
    file_out = PATH_SP / f'{indicator} {folder}' / f'{indicator} {geography}.xlsx'
    export_transit(df_mode)

    file_out = PATH_SERVER / f'{indicator} {geography}.xlsx'
    export_transit(df_mode)




In [ ]:


# Indicator
indicator = 'TravelAccess_1'
folder = 'Mode'
geography = 'Community Type'



## Employment ---
metric = 'Number of Jobs'
access = 'emp'
file_in = PATH_ACCESS / f"Community_Type_2024_dissolve__access_pop_{access}.csv"
df_emp = pd.read_csv(file_in)
df_emp = process_access_1(df_emp, geography, access, metric)
display(df_emp.head())


## Education ---
metric = 'Number of Schools'
access = 'edu'
file_in = PATH_ACCESS / f"Community_Type_2024_dissolve__access_pop_{access}.csv"
df_edu = pd.read_csv(file_in)
df_edu = process_access_1(df_edu, geography, access, metric)
display(df_edu.head())


## Neighborhood Services ---
metric = 'Number of Neighborhood Services'
access = 'nonwork'
file_in = PATH_ACCESS / f"Community_Type_2024_dissolve__access_pop_{access}.csv"
df_nonwork = pd.read_csv(file_in)
df_nonwork = process_access_1(df_nonwork, geography, access, metric)
display(df_nonwork.head())



## Merge ---

df_mode = df_emp .merge(df_edu    , on=['Reference', geography, 'Year', 'Mode'])
df_mode = df_mode.merge(df_nonwork, on=['Reference', geography, 'Year', 'Mode'])


if EXPORT:
    file_out = PATH_SP2 / f'{indicator} {folder}' / f'{indicator} {geography}.xlsx'
    export_transit(file_out, df_mode, SAMPLE_TYPE, indicator, YEAR_START, YEAR_END, geography)

    file_out = PATH_SERVER / f'{indicator} {geography}.xlsx'
    export_transit(file_out, df_mode, SAMPLE_TYPE, indicator, YEAR_START, YEAR_END, geography)





In [ ]:


# Indicator
indicator = 'TravelAccess_1'
folder = 'Mode'
geography = 'Jurisdiction'

sorted_juris = [
    'Placerville'
    , 'South Lake Tahoe'
    , 'El Dorado County'
    
    , 'Auburn'
    , 'Colfax'
    , 'Lincoln'
    , 'Loomis'
    , 'Rocklin'
    , 'Roseville'
    , 'Placer County'

    , 'Citrus Heights'
    , 'Elk Grove'
    , 'Folsom'
    , 'Galt'
    , 'Isleton'
    , 'Rancho Cordova'
    , 'Sacramento'
    , 'Sacramento County'

    , 'Live Oak'
    , 'Yuba City'
    , 'Sutter County'

    , 'Davis'
    , 'West Sacramento'
    , 'Winters'
    , 'Woodland'
    , 'Yolo County'

    , 'Marysville'
    , 'Wheatland'
    , 'Yuba County'
]



## Employment ---
metric = 'Number of Jobs'
access = 'emp'
file_in = PATH_ACCESS / f"City_County__access_pop_{access}.csv"
df_emp = pd.read_csv(file_in)
df_emp = process_access_1(df_emp, geography, access, metric)
display(df_emp.head())



## Education ---
metric = 'Number of Schools'
access = 'edu'
file_in = PATH_ACCESS / f"City_County__access_pop_{access}.csv"
df_edu = pd.read_csv(file_in)
df_edu = process_access_1(df_edu, geography, access, metric)
display(df_edu.head())



## Neighborhood Services ---
metric = 'Number of Neighborhood Services'
access = 'nonwork'
file_in = PATH_ACCESS / f"City_County__access_pop_{access}.csv"
df_nonwork = pd.read_csv(file_in)
df_nonwork = process_access_1(df_nonwork, geography, access, metric)


display(df_nonwork.head())



## Merge ---

df_mode = df_emp .merge(df_edu    , on=['Reference', geography, 'Year', 'Mode'])
df_mode = df_mode.merge(df_nonwork, on=['Reference', geography, 'Year', 'Mode'])
df_mode['Sort_mode'] = pd.Categorical(df_mode['Mode'], [
    'transit_emp'
    , 'bike_emp'
    , 'walk_emp'
])
df_mode['Sort_juris'] = pd.Categorical(df_mode['Jurisdiction'], sorted_juris)
df_mode = df_mode.sort_values(['Sort_juris', 'Sort_mode'], ascending=[True, True])
df_mode = df_mode.drop(['Sort_juris', 'Sort_mode'], axis=1)
df_mode = df_mode[df_mode['Jurisdiction'] != 'South Lake Tahoe']
df_mode = df_mode.reset_index(drop=True)
display(df_mode)

if EXPORT:
    file_out = PATH_SP / f'{indicator} {folder}' / f'{indicator} {geography}.xlsx'
    export_transit(file_out, df_mode, SAMPLE_TYPE, indicator, YEAR_START, YEAR_END, geography)

    file_out = PATH_SERVER / f'{indicator} {geography}.xlsx'
    export_transit(file_out, df_mode, SAMPLE_TYPE, indicator, YEAR_START, YEAR_END, geography)





In [ ]:



indicator = 'TravelAccess_1'
folder = 'Mode'
geography = 'County'


## Employment ---
metric = 'Number of Jobs'
access = 'emp'
file_in = PATH_ACCESS / f"tl_2020_sacog_county__access_pop_{access}.csv"
df_emp = pd.read_csv(file_in)
df_emp = process_access_1(df_emp, geography, access, metric)
display(df_emp.head())



## Education ---
metric = 'Number of Schools'
access = 'edu'
file_in = PATH_ACCESS / f"tl_2020_sacog_county__access_pop_{access}.csv"
df_edu = pd.read_csv(file_in)
df_edu = process_access_1(df_edu, geography, access, metric)
display(df_edu.head())



## Neighborhood Services ---
metric = 'Number of Neighborhood Services'
access = 'nonwork'
file_in = PATH_ACCESS / f"tl_2020_sacog_county__access_pop_{access}.csv"
df_nonwork = pd.read_csv(file_in)
df_nonwork = process_access_1(df_nonwork, geography, access, metric)


display(df_nonwork.head())


## Merge ---

df_mode = df_emp .merge(df_edu    , on=['Reference', geography, 'Year', 'Mode'])
df_mode = df_mode.merge(df_nonwork, on=['Reference', geography, 'Year', 'Mode'])
df_mode['Sort'] = pd.Categorical(df_mode['County'], [
    'El Dorado'
    , 'Placer'
    , 'Sacramento'
    , 'Sutter'
    , 'Yolo'
    , 'Yuba'
])
df_mode = df_mode.sort_values(['Sort'], ascending=[True])
df_mode = df_mode.drop(['Sort'], axis=1)
df_mode = df_mode.reset_index(drop=True)


if EXPORT:
    file_out = PATH_SP / f'{indicator} {folder}' / f'{indicator} {geography}.xlsx'
    export_transit(file_out, df_mode, SAMPLE_TYPE, indicator, YEAR_START, YEAR_END, geography)

    file_out = PATH_SERVER / f'{indicator} {geography}.xlsx'
    export_transit(file_out, df_mode, SAMPLE_TYPE, indicator, YEAR_START, YEAR_END, geography)





TravelAccess_2

In [ ]:


indicator = 'TravelAccess_2'
folder = 'Race'
geography = 'MPO'


## Employment ---

access = 'emp'
metric = 'Number of Jobs'

file_in1 = PATH_ACCESS / f"SACOG_MPO__access_asian_{access}.csv"
file_in2 = PATH_ACCESS / f"SACOG_MPO__access_black_{access}.csv"
file_in3 = PATH_ACCESS / f"SACOG_MPO__access_white_{access}.csv"
file_in4 = PATH_ACCESS / f"SACOG_MPO__access_hispanic_{access}.csv"

df_emp1 = pd.read_csv(file_in1)
df_emp2 = pd.read_csv(file_in2)
df_emp3 = pd.read_csv(file_in3)
df_emp4 = pd.read_csv(file_in4)

df_emp1['Race_Ethnicity'] = 'Asian (NH)'
df_emp2['Race_Ethnicity'] = 'Black or African American (NH)'
df_emp3['Race_Ethnicity'] = 'White (NH)'
df_emp4['Race_Ethnicity'] = 'Hispanic or Latino'

df_emp = pd.concat([df_emp1, df_emp2, df_emp3, df_emp4])

df_emp = process_access_2(df_emp, geography, access, metric)
display(df_emp.head())



## Education ---
access = 'edu'
metric = 'Number of Schools'

file_in1 = PATH_ACCESS / f"SACOG_MPO__access_asian_{access}.csv"
file_in2 = PATH_ACCESS / f"SACOG_MPO__access_black_{access}.csv"
file_in3 = PATH_ACCESS / f"SACOG_MPO__access_white_{access}.csv"
file_in4 = PATH_ACCESS / f"SACOG_MPO__access_hispanic_{access}.csv"

df_edu1 = pd.read_csv(file_in1)
df_edu2 = pd.read_csv(file_in2)
df_edu3 = pd.read_csv(file_in3)
df_edu4 = pd.read_csv(file_in4)

df_edu1['Race_Ethnicity'] = 'Asian (NH)'
df_edu2['Race_Ethnicity'] = 'Black or African American (NH)'
df_edu3['Race_Ethnicity'] = 'White (NH)'
df_edu4['Race_Ethnicity'] = 'Hispanic or Latino'

df_edu = pd.concat([df_edu1, df_edu2, df_edu3, df_edu4])

df_edu = process_access_2(df_edu, geography, access, metric)
display(df_edu.head())




## Neighborhood Services ---
access = 'nonwork'
metric = 'Number of Neighborhood Services'

file_in1 = PATH_ACCESS / f"SACOG_MPO__access_asian_{access}.csv"
file_in2 = PATH_ACCESS / f"SACOG_MPO__access_black_{access}.csv"
file_in3 = PATH_ACCESS / f"SACOG_MPO__access_white_{access}.csv"
file_in4 = PATH_ACCESS / f"SACOG_MPO__access_hispanic_{access}.csv"

df_nonwork1 = pd.read_csv(file_in1)
df_nonwork2 = pd.read_csv(file_in2)
df_nonwork3 = pd.read_csv(file_in3)
df_nonwork4 = pd.read_csv(file_in4)

df_nonwork1['Race_Ethnicity'] = 'Asian (NH)'
df_nonwork2['Race_Ethnicity'] = 'Black or African American (NH)'
df_nonwork3['Race_Ethnicity'] = 'White (NH)'
df_nonwork4['Race_Ethnicity'] = 'Hispanic or Latino'
df_nonwork = pd.concat([df_nonwork1, df_nonwork2, df_nonwork3, df_nonwork4])
df_nonwork = process_access_2(df_nonwork, geography, access, metric)
display(df_nonwork.head())


## Merge ---

df_mode = df_emp .merge(df_edu    , on=['Reference', 'Race_Ethnicity', geography, 'Year', 'Mode'])
df_mode = df_mode.merge(df_nonwork, on=['Reference', 'Race_Ethnicity', geography, 'Year', 'Mode'])

display(df_mode)

if EXPORT:
    file_out = PATH_SP / f'{indicator} {folder}' / f'{indicator} {geography}.xlsx'
    export_transit(file_out, df_mode, SAMPLE_TYPE, indicator, YEAR_START, YEAR_END, geography)

    file_out = PATH_SERVER / f'{indicator} {geography}.xlsx'
    export_transit(file_out, df_mode, SAMPLE_TYPE, indicator, YEAR_START, YEAR_END, geography)



In [ ]:


indicator = 'TravelAccess_2'
folder = 'Race'
geography = 'County'


## Employment ---
access = 'emp'
metric = 'Number of Jobs'

file_in1 = PATH_ACCESS / f"tl_2020_sacog_county__access_asian_{access}.csv"
file_in2 = PATH_ACCESS / f"tl_2020_sacog_county__access_black_{access}.csv"
file_in3 = PATH_ACCESS / f"tl_2020_sacog_county__access_white_{access}.csv"
file_in4 = PATH_ACCESS / f"tl_2020_sacog_county__access_hispanic_{access}.csv"

df_emp1 = pd.read_csv(file_in1)
df_emp2 = pd.read_csv(file_in2)
df_emp3 = pd.read_csv(file_in3)
df_emp4 = pd.read_csv(file_in4)

df_emp1['Race_Ethnicity'] = 'Asian (NH)'
df_emp2['Race_Ethnicity'] = 'Black or African American (NH)'
df_emp3['Race_Ethnicity'] = 'White (NH)'
df_emp4['Race_Ethnicity'] = 'Hispanic or Latino'

df_emp = pd.concat([df_emp1, df_emp2, df_emp3, df_emp4])

df_emp = process_access_2(df_emp, geography, access, metric)
display(df_emp.head())



## Education ---
access = 'edu'
metric = 'Number of Schools'

file_in1 = PATH_ACCESS / f"tl_2020_sacog_county__access_asian_{access}.csv"
file_in2 = PATH_ACCESS / f"tl_2020_sacog_county__access_black_{access}.csv"
file_in3 = PATH_ACCESS / f"tl_2020_sacog_county__access_white_{access}.csv"
file_in4 = PATH_ACCESS / f"tl_2020_sacog_county__access_hispanic_{access}.csv"

df_edu1 = pd.read_csv(file_in1)
df_edu2 = pd.read_csv(file_in2)
df_edu3 = pd.read_csv(file_in3)
df_edu4 = pd.read_csv(file_in4)

df_edu1['Race_Ethnicity'] = 'Asian (NH)'
df_edu2['Race_Ethnicity'] = 'Black or African American (NH)'
df_edu3['Race_Ethnicity'] = 'White (NH)'
df_edu4['Race_Ethnicity'] = 'Hispanic or Latino'

df_edu = pd.concat([df_edu1, df_edu2, df_edu3, df_edu4])

df_edu = process_access_2(df_edu, geography, access, metric)
display(df_edu.head())




## Neighborhood Services ---
access = 'nonwork'
metric = 'Number of Neighborhood Services'

file_in1 = PATH_ACCESS / f"tl_2020_sacog_county__access_asian_{access}.csv"
file_in2 = PATH_ACCESS / f"tl_2020_sacog_county__access_black_{access}.csv"
file_in3 = PATH_ACCESS / f"tl_2020_sacog_county__access_white_{access}.csv"
file_in4 = PATH_ACCESS / f"tl_2020_sacog_county__access_hispanic_{access}.csv"

df_nonwork1 = pd.read_csv(file_in1)
df_nonwork2 = pd.read_csv(file_in2)
df_nonwork3 = pd.read_csv(file_in3)
df_nonwork4 = pd.read_csv(file_in4)

df_nonwork1['Race_Ethnicity'] = 'Asian (NH)'
df_nonwork2['Race_Ethnicity'] = 'Black or African American (NH)'
df_nonwork3['Race_Ethnicity'] = 'White (NH)'
df_nonwork4['Race_Ethnicity'] = 'Hispanic or Latino'


df_nonwork = pd.concat([df_nonwork1, df_nonwork2, df_nonwork3, df_nonwork4])

df_nonwork = process_access_2(df_nonwork, geography, access, metric)
display(df_nonwork.head())


## Merge ---

df_mode = df_emp .merge(df_edu    , on=['Reference', 'Race_Ethnicity', geography, 'Year', 'Mode'])
df_mode = df_mode.merge(df_nonwork, on=['Reference', 'Race_Ethnicity', geography, 'Year', 'Mode'])

display(df_mode)

if EXPORT:
    file_out = PATH_SP / f'{indicator} {folder}' / f'{indicator} {geography}.xlsx'
    export_transit(file_out, df_mode, SAMPLE_TYPE, indicator, YEAR_START, YEAR_END, geography)

    file_out = PATH_SERVER / f'{indicator} {geography}.xlsx'
    export_transit(file_out, df_mode, SAMPLE_TYPE, indicator, YEAR_START, YEAR_END, geography)



In [ ]:


indicator = 'TravelAccess_2'
folder = 'Race'
geography = 'Jurisdiction'


sorted_juris = [
    'Placerville'
    , 'South Lake Tahoe'
    , 'El Dorado County'
    
    , 'Auburn'
    , 'Colfax'
    , 'Lincoln'
    , 'Loomis'
    , 'Rocklin'
    , 'Roseville'
    , 'Placer County'

    , 'Citrus Heights'
    , 'Elk Grove'
    , 'Folsom'
    , 'Galt'
    , 'Isleton'
    , 'Rancho Cordova'
    , 'Sacramento'
    , 'Sacramento County'

    , 'Live Oak'
    , 'Yuba City'
    , 'Sutter County'

    , 'Davis'
    , 'West Sacramento'
    , 'Winters'
    , 'Woodland'
    , 'Yolo County'

    , 'Marysville'
    , 'Wheatland'
    , 'Yuba County'
]



## Employment ---

access = 'emp'
metric = 'Number of Jobs'

file_in1 = PATH_ACCESS / f"City_County__access_asian_{access}.csv"
file_in2 = PATH_ACCESS / f"City_County__access_black_{access}.csv"
file_in3 = PATH_ACCESS / f"City_County__access_white_{access}.csv"
file_in4 = PATH_ACCESS / f"City_County__access_hispanic_{access}.csv"

df_emp1 = pd.read_csv(file_in1)
df_emp2 = pd.read_csv(file_in2)
df_emp3 = pd.read_csv(file_in3)
df_emp4 = pd.read_csv(file_in4)

df_emp1['Race_Ethnicity'] = 'Asian (NH)'
df_emp2['Race_Ethnicity'] = 'Black or African American (NH)'
df_emp3['Race_Ethnicity'] = 'White (NH)'
df_emp4['Race_Ethnicity'] = 'Hispanic or Latino'

df_emp = pd.concat([df_emp1, df_emp2, df_emp3, df_emp4])

df_emp = process_access_2(df_emp, geography, access, metric)
display(df_emp.head())



## Education ---
access = 'edu'
metric = 'Number of Schools'

file_in1 = PATH_ACCESS / f"City_County__access_asian_{access}.csv"
file_in2 = PATH_ACCESS / f"City_County__access_black_{access}.csv"
file_in3 = PATH_ACCESS / f"City_County__access_white_{access}.csv"
file_in4 = PATH_ACCESS / f"City_County__access_hispanic_{access}.csv"

df_edu1 = pd.read_csv(file_in1)
df_edu2 = pd.read_csv(file_in2)
df_edu3 = pd.read_csv(file_in3)
df_edu4 = pd.read_csv(file_in4)

df_edu1['Race_Ethnicity'] = 'Asian (NH)'
df_edu2['Race_Ethnicity'] = 'Black or African American (NH)'
df_edu3['Race_Ethnicity'] = 'White (NH)'
df_edu4['Race_Ethnicity'] = 'Hispanic or Latino'

df_edu = pd.concat([df_edu1, df_edu2, df_edu3, df_edu4])

df_edu = process_access_2(df_edu, geography, access, metric)
display(df_edu.head())




## Neighborhood Services ---
access = 'nonwork'
metric = 'Number of Neighborhood Services'

file_in1 = PATH_ACCESS / f"City_County__access_asian_{access}.csv"
file_in2 = PATH_ACCESS / f"City_County__access_black_{access}.csv"
file_in3 = PATH_ACCESS / f"City_County__access_white_{access}.csv"
file_in4 = PATH_ACCESS / f"City_County__access_hispanic_{access}.csv"

df_nonwork1 = pd.read_csv(file_in1)
df_nonwork2 = pd.read_csv(file_in2)
df_nonwork3 = pd.read_csv(file_in3)
df_nonwork4 = pd.read_csv(file_in4)

df_nonwork1['Race_Ethnicity'] = 'Asian (NH)'
df_nonwork2['Race_Ethnicity'] = 'Black or African American (NH)'
df_nonwork3['Race_Ethnicity'] = 'White (NH)'
df_nonwork4['Race_Ethnicity'] = 'Hispanic or Latino'
df_nonwork = pd.concat([df_nonwork1, df_nonwork2, df_nonwork3, df_nonwork4])
df_nonwork = process_access_2(df_nonwork, geography, access, metric)
display(df_nonwork.head())


## Merge ---

df_mode = df_emp .merge(df_edu    , on=['Reference', 'Race_Ethnicity', geography, 'Year', 'Mode'])
df_mode = df_mode.merge(df_nonwork, on=['Reference', 'Race_Ethnicity', geography, 'Year', 'Mode'])
df_mode['Sort_mode'] = pd.Categorical(df_mode['Mode'], [
    'transit_emp'
    , 'bike_emp'
    , 'walk_emp'
])
df_mode['Sort_juris'] = pd.Categorical(df_mode['Jurisdiction'], sorted_juris)
df_mode = df_mode.sort_values(['Sort_juris', 'Sort_mode'], ascending=[True, True])
df_mode = df_mode.drop(['Sort_juris', 'Sort_mode'], axis=1)
df_mode = df_mode[df_mode['Jurisdiction'] != 'South Lake Tahoe']
df_mode = df_mode.reset_index(drop=True)

display(df_mode)

if EXPORT:
    file_out = PATH_SP / f'{indicator} {folder}' / f'{indicator} {geography}.xlsx'
    export_transit(file_out, df_mode, SAMPLE_TYPE, indicator, YEAR_START, YEAR_END, geography)

    file_out = PATH_SERVER / f'{indicator} {geography}.xlsx'
    export_transit(file_out, df_mode, SAMPLE_TYPE, indicator, YEAR_START, YEAR_END, geography)



In [ ]:


indicator = 'TravelAccess_2'
folder = 'Race'
geography = 'Community Type'


## Employment ---

access = 'emp'
metric = 'Number of Jobs'

file_in1 = PATH_ACCESS / f"Community_Type_2024_dissolve__access_asian_{access}.csv"
file_in2 = PATH_ACCESS / f"Community_Type_2024_dissolve__access_black_{access}.csv"
file_in3 = PATH_ACCESS / f"Community_Type_2024_dissolve__access_white_{access}.csv"
file_in4 = PATH_ACCESS / f"Community_Type_2024_dissolve__access_hispanic_{access}.csv"

df_emp1 = pd.read_csv(file_in1)
df_emp2 = pd.read_csv(file_in2)
df_emp3 = pd.read_csv(file_in3)
df_emp4 = pd.read_csv(file_in4)

df_emp1['Race_Ethnicity'] = 'Asian (NH)'
df_emp2['Race_Ethnicity'] = 'Black or African American (NH)'
df_emp3['Race_Ethnicity'] = 'White (NH)'
df_emp4['Race_Ethnicity'] = 'Hispanic or Latino'

df_emp = pd.concat([df_emp1, df_emp2, df_emp3, df_emp4])

df_emp = process_access_2(df_emp, geography, access, metric)
display(df_emp.head())



## Education ---
access = 'edu'
metric = 'Number of Schools'

file_in1 = PATH_ACCESS / f"Community_Type_2024_dissolve__access_asian_{access}.csv"
file_in2 = PATH_ACCESS / f"Community_Type_2024_dissolve__access_black_{access}.csv"
file_in3 = PATH_ACCESS / f"Community_Type_2024_dissolve__access_white_{access}.csv"
file_in4 = PATH_ACCESS / f"Community_Type_2024_dissolve__access_hispanic_{access}.csv"

df_edu1 = pd.read_csv(file_in1)
df_edu2 = pd.read_csv(file_in2)
df_edu3 = pd.read_csv(file_in3)
df_edu4 = pd.read_csv(file_in4)

df_edu1['Race_Ethnicity'] = 'Asian (NH)'
df_edu2['Race_Ethnicity'] = 'Black or African American (NH)'
df_edu3['Race_Ethnicity'] = 'White (NH)'
df_edu4['Race_Ethnicity'] = 'Hispanic or Latino'

df_edu = pd.concat([df_edu1, df_edu2, df_edu3, df_edu4])

df_edu = process_access_2(df_edu, geography, access, metric)
display(df_edu.head())




## Neighborhood Services ---
access = 'nonwork'
metric = 'Number of Neighborhood Services'

file_in1 = PATH_ACCESS / f"Community_Type_2024_dissolve__access_asian_{access}.csv"
file_in2 = PATH_ACCESS / f"Community_Type_2024_dissolve__access_black_{access}.csv"
file_in3 = PATH_ACCESS / f"Community_Type_2024_dissolve__access_white_{access}.csv"
file_in4 = PATH_ACCESS / f"Community_Type_2024_dissolve__access_hispanic_{access}.csv"

df_nonwork1 = pd.read_csv(file_in1)
df_nonwork2 = pd.read_csv(file_in2)
df_nonwork3 = pd.read_csv(file_in3)
df_nonwork4 = pd.read_csv(file_in4)

df_nonwork1['Race_Ethnicity'] = 'Asian (NH)'
df_nonwork2['Race_Ethnicity'] = 'Black or African American (NH)'
df_nonwork3['Race_Ethnicity'] = 'White (NH)'
df_nonwork4['Race_Ethnicity'] = 'Hispanic or Latino'
df_nonwork = pd.concat([df_nonwork1, df_nonwork2, df_nonwork3, df_nonwork4])
df_nonwork = process_access_2(df_nonwork, geography, access, metric)
display(df_nonwork.head())


## Merge ---

df_mode = df_emp .merge(df_edu    , on=['Reference', 'Race_Ethnicity', geography, 'Year', 'Mode'])
df_mode = df_mode.merge(df_nonwork, on=['Reference', 'Race_Ethnicity', geography, 'Year', 'Mode'])
display(df_mode)

if EXPORT:
    file_out = PATH_SP / f'{indicator} {folder}' / f'{indicator} {geography}.xlsx'
    export_transit(file_out, df_mode, SAMPLE_TYPE, indicator, YEAR_START, YEAR_END, geography)

    file_out = PATH_SERVER / f'{indicator} {geography}.xlsx'
    export_transit(file_out, df_mode, SAMPLE_TYPE, indicator, YEAR_START, YEAR_END, geography)

